In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path(r"C:\Users\dparh\Desktop\ML\Assignments\Assignment-1\data")

sns.set_style("whitegrid")
sns.set_palette("viridis")

df = pd.read_csv(DATA_PATH / "jardin_data.csv")

df.head(5)

In [ ]:
df.describe()

In [ ]:
numeric = df.select_dtypes('number')

numeric.columns

In [ ]:
target = "Caudel" if "Caudel" in numeric.columns else "Caudal"

features = (
    numeric.corr()[target]
    .drop(target)
    .abs()
    .nlargest(6)
    .index
)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for feature, ax in zip(features, axes.flat):
    sns.scatterplot(data=numeric, x=feature, y=target, alpha=0.4, s=18, ax=ax)
    ax.set_title(f"{feature} vs. {target}")

plt.tight_layout()
plt.show()

In [ ]:
corr_mtx = numeric.corr()

corr_mtx

In [ ]:
season_cols = df.filter(regex=r"^Season_").columns
df[season_cols] = df[season_cols].astype(int)

Time to implement some models

In [ ]:
# start with a simple linear regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, ElasticNet, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

X = df.drop(columns=['Fecha', 'Caudal'], inplace=False)
y = df['Caudal']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=2026)

In [ ]:
X

In [ ]:
y

Simple Linear Regression

In [ ]:
model_linear = LinearRegression().fit(X_train, y_train)

predictions_linear = model_linear.predict(X_test)

mse = mean_squared_error(y_test, predictions_linear)
rmse = np.sqrt(mse)

print("##### Linear regression performance #####")
print(f"R² score: {r2_score(y_test, predictions_linear):.4f}")
print(f"MSE:      {mse:.4f}")
print(f"RMSE:     {rmse:.4f}")
print(f"Intercept: {model_linear.intercept_:.4f}")

coefficients = pd.Series(
    model_linear.coef_,
    index=X_train.columns,
    name="coefficient"
).sort_values(key=abs, ascending=False)

display(coefficients)

Ridge regression

In [ ]:
# ! Ridge requires scaling for proper realisation
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Fecha', 'Caudal'], inplace=False)
y = df['Caudal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train_scaled, y_train)

predictions_ridge = model_ridge.predict(X_test_scaled)

# Performance measure
print(f"Coefficients (Ridge): {model_ridge.coef_}")
print("########################")
print(f"Intercept (Ridge): {model_ridge.intercept_}")
print(f"R squared score (Ridge): {model_ridge.score(X_test_scaled, y_test):.4f}")
print("########################")

In [ ]:
# now tune the Ridge model with CV
param_grid = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
model_ridge = Ridge(fit_intercept=True, copy_X=True, positive=False)

configs = model_ridge.get_params()
alphas_test = np.logspace(-4,4,100)

ridge_cv = RidgeCV(
    alphas=param_grid,
    cv=None,
    fit_intercept=True
)

ridge_cv.fit(X_train_scaled, y_train)
print(f"Optimal lambda fot the Ridge model: {ridge_cv.alpha_}")

In [ ]:
model_ridge_optimal = Ridge(alpha=100, max_iter=1000)
model_ridge_optimal.fit(X_train_scaled, y_train)
predictions_ridge_optimal = model_ridge_optimal.predict(X_test_scaled)

# Performance measure
print(f"Coefficients (Ridge optimal): {model_ridge_optimal.coef_}")
print("########################")
print(f"Intercept (Ridge optimal): {model_ridge_optimal.intercept_}")
print(f"R squared score (Ridge optimal): {model_ridge_optimal.score(X_test_scaled, y_test):.4f}")
print(f"RMSE for (Ridge optimal): {mean_squared_error(y_test, predictions_ridge_optimal):.4f}")
print("########################")

Lasso model

In [ ]:
model_lasso = Lasso(max_iter=1000)

X = df.drop(columns=['Fecha', 'Caudal'], inplace=False)
y = df['Caudal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

model_lasso.fit(X_train_scaled, y_train)
predictions_lasso = model_lasso.predict(X_test_scaled)

print(f"Coefficients (Lasso): {model_lasso.coef_}")
print("########################")
print(f"Intercept (Lasso): {model_lasso.intercept_}")
print(f"R squared score (Lasso): {model_lasso.score(X_test_scaled, y_test):.4f}")
print(f"RMSE for (Lasso): {mean_squared_error(y_test, predictions_lasso):.4f}")
print("########################")

In [ ]:
# tune the Lasso model with LassCV
model_lasso_cv = LassoCV(cv=5, random_state=2026)

model_lasso_cv.fit(X_train_scaled, y_train)
predictions_lasso_cv = model_lasso_cv.predict(X_test_scaled)

# Performance measure
print(f"Optimal alpha (LassoCV): {model_lasso_cv.alpha_}")
print(f"Coefficients (LassoCV): {model_lasso_cv.coef_}")
print(f"R squared score (LassoCV): {model_lasso_cv.score(X_test_scaled, y_test):.4f}")
print(f"RMSE for (LassoCV): {mean_squared_error(y_test, predictions_lasso_cv):.4f}")

Elastic Net model

In [ ]:
# Elastic Net with cross-validated hyperparameter tuning
X = df.drop(columns=["Fecha", "Caudal"])
y = df["Caudal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_elastic_net_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-4, 2, 100),
    cv=5,
    max_iter=20000,
    random_state=2026,
    n_jobs=-1
)

model_elastic_net_cv.fit(X_train_scaled, y_train)
predictions_elastic_net = model_elastic_net_cv.predict(X_test_scaled)

print(f"Best alpha:   {model_elastic_net_cv.alpha_:.6g}")
print(f"Best l1_ratio: {model_elastic_net_cv.l1_ratio_:.2f}")
print(f"R²:           {r2_score(y_test, predictions_elastic_net):.4f}")
print(f"RMSE:         {np.sqrt(mean_squared_error(y_test, predictions_elastic_net)):.4f}")

elastic_coefficients = pd.Series(
    model_elastic_net_cv.coef_,
    index=X_train.columns,
    name="coefficient"
).sort_values(key=abs, ascending=False)

display(elastic_coefficients)

In [ ]:
def choose_best_model(models, X_test_scaled, y_test):
    best_model = None
    best_score = -np.inf
    
    for model in models:
        score = model.score(X_test_scaled, y_test)
        if score > best_score:
            best_score = score
            best_model = model
    print(f"Found best model: {best_model.__class__.__name__} with R² score: {best_score:.4f}")
    return best_model

print(choose_best_model([model_linear, model_ridge_optimal, model_lasso_cv, model_elastic_net_cv], X_test_scaled, y_test))

Lasso Wins